# 08 · Noticias y bolsa

**Objetivo:** medir si el sentimiento de las noticias de una empresa se relaciona con lo que hace su acción después. Los tres modelos del notebook 07 (M1 diccionario, M2 Transformer y M3 BERT) puntúan los titulares de FNSPID que mencionan a su empresa, y se compara la rentabilidad de la acción, menos la del mercado, después de las noticias positivas, neutrales y negativas. Antes se comprueba con 300 titulares recientes etiquetados a mano si los modelos, entrenados con frases de PhraseBank, también saben leer titulares.

**Plan:**

| Paso | Qué se hace | Estado |
|---|---|---|
| 1 | Plantilla de 300 titulares de NewsAPI para etiquetar a mano, con la regla de etiquetado | hecho |
| 2 | Los tres modelos del 07: M1 desde su fichero; M2 y M3 con sus capas, copiadas en `capas_keras.py` | hecho |
| 3 | M1, M2 y M3 puntúan los titulares de FNSPID que mencionan a su empresa | hecho |
| 4 | Los tres modelos con los 300 titulares etiquetados a mano | pendiente |
| 5 | SPY y rentabilidad de cada acción menos la del mercado | pendiente |
| 6 | Sentimiento diario por empresa y estudio de eventos | pendiente |

**Entradas:**
- `data/cache/noticias_newsapi.parquet` y `noticias_fnspid.parquet` (notebook 06)
- `data/cache/empresas_sec.parquet` (notebook 03): nombre de cada empresa
- `etiquetas/titulares_a_mano.csv`: los 300 titulares, etiquetados a mano por el autor
- `data/modelos/` (notebook 07): `vocabulario.txt`, `m1_diccionario.json`, `m2_eleccion.json`, `m2_sin_pesos.keras` y `m3_mezclado_descongelado.keras` (M3, el corregido)
- `data/cache/predicciones_phrasebank.parquet` (notebook 07): para comprobar que los modelos se cargan bien

**Salidas:**
- `etiquetas/titulares_a_mano.csv`: la plantilla (solo si no existe, o se amplía por el final; nunca se borra una etiqueta)
- `data/cache/sentimiento_titulares.parquet`: el sentimiento de cada titular de FNSPID según cada modelo (una fila por titular y modelo)

In [1]:
import json
import re
import warnings
from pathlib import Path

from capas_keras import CAPAS_PROPIAS, cargar_vectorizador   # va primero: fija el motor de Keras (TensorFlow)
import keras
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from funciones import (
    CARPETA_CACHE,
    CLASES_SENTIMIENTO,
    SEMILLA,
    clasificar_tono,
    tono_loughran_mcdonald,
)

load_dotenv()
print("Keras", keras.__version__, "· motor:", keras.backend.backend())

Keras 3.15.1 · motor: tensorflow


## 1. ¿Saben los modelos leer titulares? La plantilla para etiquetar a mano

Los modelos del 07 se entrenaron y se examinaron con frases de **notas de prensa de empresas finlandesas de 2004 a 2010** (Financial PhraseBank). Aquí se aplican a **titulares de noticias de empresas de EEUU**, más cortos y con otra jerga. Si en este notebook el sentimiento no se relacionara con la bolsa, habría que saber si es porque no hay relación o porque los modelos no saben leer titulares. No existe un conjunto de titulares de nuestras empresas con su sentimiento etiquetado, así que el autor etiqueta **300 a mano**. Al principio se previeron 200; antes de etiquetar ninguno y de ver ningún resultado, el autor amplió a 300 para medir con más precisión (el intervalo del F1 pasa de unos ±0,065 a unos ±0,055).

**Qué titulares:** los recientes de NewsAPI (notebook 06), que ya mencionan a su empresa. Son de otra época que los de FNSPID, así que es una prueba **fuera de dominio** en dos sentidos: otro tipo de texto que PhraseBank y otras fechas. Se eligen al azar con la semilla del proyecto, sin titulares repetidos y **como mucho 5 por empresa**, para que unas pocas empresas muy citadas (Amazon tiene el 14 % de los titulares) no se lleven la muestra. Para que la muestra sea siempre la misma aunque se vuelva a ejecutar el 06, se usan solo los titulares descargados hasta el 16 nov 2026, el día en que se hizo.

**La regla de etiquetado** se escribió antes de empezar, en `etiquetas/regla_de_etiquetado.md`. Es la misma pregunta que se hizo a los anotadores de PhraseBank: desde el punto de vista de un inversor en esa empresa, ¿el titular es una noticia buena (`positive`), mala (`negative`) o neutra (`neutral`) para el precio de su acción? Solo con el titular, que es lo que ven los modelos; en la duda, `neutral`; y sin mirar lo que dicen los modelos.

**Titulares que no tratan de la empresa.** El filtro del 06 (`menciona_empresa`) acepta un titular si contiene el ticker o el nombre de la empresa, y a veces coinciden con una palabra corriente (*principal*, *hinge*, *holding*, *SNAP*). Esos titulares no se quitan: también están entre los de FNSPID, así que forman parte de lo que leerán los modelos. Se marcan en una columna, `trata_de_la_empresa` (`sí` o `no`), y se etiquetan como `neutral`, porque no deberían mover la acción. Así se pueden dar dos resultados: con los 300 titulares (lo que pasará en FNSPID) y solo con los que tratan de la empresa (cómo leen los modelos un titular de verdad). Y se mide de paso cuántas veces se equivoca el filtro.

In [2]:
CARPETA_ETIQUETAS = Path("etiquetas")
RUTA_ETIQUETADO = CARPETA_ETIQUETAS / "titulares_a_mano.csv"
TITULARES_PREVISTOS = 200   # la primera plantilla
TITULARES_A_MANO = 300      # ampliada antes de etiquetar: los 100 nuevos van detrás, con id 201-300
MAXIMO_POR_EMPRESA = 5
FECHA_MUESTRA = pd.Timestamp("2026-11-16")   # titulares descargados hasta el día en que se hizo la muestra

noticias_recientes = pd.read_parquet(CARPETA_CACHE / "noticias_newsapi.parquet")
noticias_recientes = noticias_recientes[noticias_recientes["descargado"] <= FECHA_MUESTRA]
print(f"Titulares de NewsAPI: {len(noticias_recientes):,} · distintos: {noticias_recientes['titular'].nunique():,}"
      f" · empresas: {noticias_recientes['ticker'].nunique()}"
      f" · del {noticias_recientes['fecha'].min():%d %b %Y} al {noticias_recientes['fecha'].max():%d %b %Y}")

Titulares de NewsAPI: 3,036 · distintos: 2,671 · empresas: 106 · del 20 Sep 2026 al 15 Nov 2026


La muestra: se barajan los titulares distintos con la semilla, se quedan los 5 primeros de cada empresa y, de ellos, se eligen 200 al azar. Los 100 de la ampliación se eligen igual entre los candidatos que quedan, así que los 200 primeros no cambian. Se añade el nombre de la empresa para saber de cuál se juzga cada titular.


In [3]:
nombres = pd.read_parquet(CARPETA_CACHE / "empresas_sec.parquet").set_index("ticker")["name"]

candidatos = (noticias_recientes.drop_duplicates("titular")
              .sample(frac=1, random_state=SEMILLA)          # orden al azar, con semilla
              .groupby("ticker").head(MAXIMO_POR_EMPRESA))   # como mucho 5 por empresa
primeros = candidatos.sample(TITULARES_PREVISTOS, random_state=SEMILLA)
ampliacion = candidatos.drop(primeros.index).sample(TITULARES_A_MANO - TITULARES_PREVISTOS, random_state=SEMILLA)
muestra = pd.concat([primeros, ampliacion])

plantilla = pd.DataFrame({
    "id": range(1, TITULARES_A_MANO + 1),
    "ticker": muestra["ticker"].to_numpy(),
    "empresa": muestra["ticker"].map(nombres).to_numpy(),
    "titular": muestra["titular"].to_numpy(),
    "trata_de_la_empresa": "",
    "etiqueta": "",
})
print(f"Candidatos (como mucho {MAXIMO_POR_EMPRESA} por empresa): {len(candidatos)} · elegidos: {len(plantilla)}"
      f" · empresas en la muestra: {plantilla['ticker'].nunique()}")
plantilla.head()

Candidatos (como mucho 5 por empresa): 457 · elegidos: 300 · empresas en la muestra: 105


,id,ticker,empresa,titular,trata_de_la_empresa,etiqueta
0,1,KMI,"KINDER MORGAN, INC.",Energy Transfer vs. Kinder Morgan: Which Natur...,,
1,2,PFG,PRINCIPAL FINANCIAL GROUP INC,School principal documents Indonesia's worst w...,,
2,3,DDOG,"Datadog, Inc.",Here Are Monday’s Top Wall Street Analyst Rese...,,
3,4,TMUS,"T-Mobile US, Inc.",Do You Like Cash? Consider T-Mobile Stock,,
4,5,PENG,"Penguin Solutions, Inc.",Penguin Solutions Trumpets AI-Driven Beat-And-...,,


La plantilla se escribe **solo si no existe**. Si ya existe con menos titulares (la de 200), se le añaden por el final los que faltan, conservando lo ya etiquetado. Volver a ejecutar el notebook nunca borra una etiqueta.


In [4]:
if not RUTA_ETIQUETADO.exists():
    CARPETA_ETIQUETAS.mkdir(exist_ok=True)
    plantilla.to_csv(RUTA_ETIQUETADO, index=False, encoding="utf-8")
    print(f"Plantilla escrita: {RUTA_ETIQUETADO} ({len(plantilla)} titulares)")
else:
    actual = pd.read_csv(RUTA_ETIQUETADO, sep=None, engine="python", keep_default_na=False)
    if len(actual) < len(plantilla):
        # Las filas que ya hay tienen que ser las primeras de la plantilla: solo se añaden las que faltan
        assert actual[["id", "ticker", "titular"]].equals(plantilla[["id", "ticker", "titular"]].head(len(actual)))
        pd.concat([actual, plantilla.iloc[len(actual):]], ignore_index=True).to_csv(RUTA_ETIQUETADO, index=False, encoding="utf-8")
        print(f"Plantilla ampliada: de {len(actual)} a {len(plantilla)} titulares, sin tocar los que ya había")
    else:
        print(f"{RUTA_ETIQUETADO} ya existe con {len(actual)} titulares: no se cambia")


etiquetas/titulares_a_mano.csv ya existe con 300 titulares: no se cambia


Cuántos titulares hay etiquetados. El fichero se lee de forma flexible (con comas o con punto y coma, que es como lo guardan algunas hojas de cálculo) y sin tener en cuenta mayúsculas ni espacios (*si* vale como *sí*). Avisa de las filas a medias o mal escritas y de las que no tratan de la empresa pero no son `neutral`.

In [5]:
etiquetado = pd.read_csv(RUTA_ETIQUETADO, sep=None, engine="python", keep_default_na=False)
etiquetado["etiqueta"] = etiquetado["etiqueta"].str.strip().str.lower()
etiquetado["trata_de_la_empresa"] = etiquetado["trata_de_la_empresa"].str.strip().str.lower().replace({"si": "sí"})
hechos = etiquetado["etiqueta"].isin(CLASES_SENTIMIENTO) & etiquetado["trata_de_la_empresa"].isin(["sí", "no"])
empezados = (etiquetado["etiqueta"] != "") | (etiquetado["trata_de_la_empresa"] != "")
mal_escritos = etiquetado[empezados & ~hechos]
no_neutrales_ajenos = etiquetado[(etiquetado["trata_de_la_empresa"] == "no") & (etiquetado["etiqueta"] != "neutral")]

print(f"Etiquetados: {hechos.sum()} de {len(etiquetado)}")
print("Reparto:", etiquetado.loc[hechos, "etiqueta"].value_counts().to_dict(),
      "· no tratan de la empresa:", (etiquetado.loc[hechos, "trata_de_la_empresa"] == "no").sum())
if len(mal_escritos):
    print("A medias o mal escritos (etiqueta: positive, negative o neutral; trata_de_la_empresa: sí o no):")
    display(mal_escritos[["id", "titular", "trata_de_la_empresa", "etiqueta"]])
if len(no_neutrales_ajenos):
    print("No tratan de la empresa, pero su etiqueta no es neutral:")
    display(no_neutrales_ajenos[["id", "titular", "trata_de_la_empresa", "etiqueta"]])


Etiquetados: 300 de 300
Reparto: {'neutral': 173, 'positive': 72, 'negative': 55} · no tratan de la empresa: 29


## 2. Los tres modelos del notebook 07

Se cargan tal como quedaron en el 07, sin volver a entrenar nada:

- **M1, diccionario:** las dos listas de palabras y el umbral de "neutral" elegido en la validación (`m1_diccionario.json`).
- **M2, Transformer:** la versión elegida en la validación, sin pesos de clase (`m2_eleccion.json`).
- **M3, BERT:** el **corregido** (sección 10 del 07), que es el oficial.

Para cargar M2 y M3, Keras necesita las clases de sus capas propias (`TransformerEncoder` y `PositionalEmbedding`) y la misma estandarización del texto. Están copiadas del 07 en `capas_keras.py`: el 07 conserva las suyas para no perder sus entrenamientos, y van en un archivo aparte de `funciones.py` porque necesitan Keras, que los demás notebooks no usan.

**Comprobación de que la copia es fiel:** los tres modelos tienen que repetir exactamente lo que dijeron en el 07 sobre las 967 frases de la validación de PhraseBank.

In [6]:
CARPETA_MODELOS = CARPETA_CACHE.parent / "modelos"
SENTIMIENTO_NUMERO = {"negative": -1, "neutral": 0, "positive": 1}

# Keras avisa de que TransformerEncoder "destruye la máscara"; es lo mismo que en el 07 y en los apuntes
warnings.filterwarnings("ignore", message=".*does not support masking.*")
# y, al cargar, de que sus capas no tienen método build(): tampoco lo tienen en los apuntes, y los pesos se cargan bien
# (lo comprueba la celda siguiente, que repite exactamente las predicciones del 07)
warnings.filterwarnings("ignore", message=".*does not have a `build\\(\\)` method.*")

m1 = json.loads((CARPETA_MODELOS / "m1_diccionario.json").read_text(encoding="utf-8"))
palabras_negativas, palabras_positivas = set(m1["palabras_negativas"]), set(m1["palabras_positivas"])

vectorize_layer = cargar_vectorizador(CARPETA_MODELOS / "vocabulario.txt")
version_m2 = json.loads((CARPETA_MODELOS / "m2_eleccion.json").read_text(encoding="utf-8"))["version"]
fichero_m2 = {"sin pesos de clase": "m2_sin_pesos.keras", "con pesos de clase": "m2_con_pesos.keras"}[version_m2]
m2 = keras.models.load_model(CARPETA_MODELOS / fichero_m2, custom_objects=CAPAS_PROPIAS, compile=False)
m3 = keras.models.load_model(CARPETA_MODELOS / "m3_mezclado_descongelado.keras", custom_objects=CAPAS_PROPIAS, compile=False)
print(f"M1: umbral {m1['umbral']}, {len(palabras_negativas):,} palabras negativas y {len(palabras_positivas):,} positivas"
      f" · M2: {version_m2} ({fichero_m2}) · M3: el corregido")

M1: umbral 0.0, 2,345 palabras negativas y 347 positivas · M2: sin pesos de clase (m2_sin_pesos.keras) · M3: el corregido


Cada modelo se usa con una función que recibe una lista de textos y devuelve, para cada uno, la **clase** y la **confianza**:

- **M2 y M3:** la clase con más probabilidad, y esa probabilidad como confianza (entre 1/3 y 1).
- **M1:** la clase según el tono y el umbral, como en el 07. Como confianza, la **proporción de palabras del texto que están en el diccionario**, positivas o negativas. Un titular sin ninguna tiene confianza 0: M1 lo llama neutral porque no sabe nada de él.

In [7]:
def puntuar_diccionario(textos: list[str]) -> pd.DataFrame:
    """Clase de M1 para cada texto y su confianza: la proporción de sus palabras que están en el diccionario."""
    palabras_con_tono = palabras_negativas | palabras_positivas
    clases, confianzas = [], []
    for texto in textos:
        clases.append(clasificar_tono(tono_loughran_mcdonald(texto, palabras_negativas, palabras_positivas), m1["umbral"]))
        palabras = re.findall(r"[A-Z]+", texto.upper())   # las mismas palabras que cuenta tono_loughran_mcdonald
        confianzas.append(sum(palabra in palabras_con_tono for palabra in palabras) / len(palabras) if palabras else 0.0)
    return pd.DataFrame({"clase": clases, "confianza": confianzas})


def puntuar_red(modelo, textos: list[str]) -> pd.DataFrame:
    """Clase con más probabilidad para cada texto (M2 o M3) y esa probabilidad como confianza."""
    probabilidades = modelo.predict(vectorize_layer(textos).numpy(), batch_size=256, verbose=0)
    return pd.DataFrame({"clase": [CLASES_SENTIMIENTO[numero] for numero in probabilidades.argmax(axis=1)],
                         "confianza": probabilidades.max(axis=1)})


MODELOS = {
    "M1 diccionario": puntuar_diccionario,
    "M2 Transformer": lambda textos: puntuar_red(m2, textos),
    "M3 BERT": lambda textos: puntuar_red(m3, textos),
}

In [8]:
predicciones_07 = pd.read_parquet(CARPETA_CACHE / "predicciones_phrasebank.parquet")
validacion_07 = predicciones_07[predicciones_07["parte"] == "validacion"]
columna_en_el_07 = {"M1 diccionario": "M1 diccionario", "M2 Transformer": "M2 Transformer", "M3 BERT": "M3 BERT (corregido)"}

coinciden_con_el_07 = {
    nombre: (puntuar(validacion_07["frase"].tolist())["clase"].to_numpy() == validacion_07[columna_en_el_07[nombre]].to_numpy()).mean()
    for nombre, puntuar in MODELOS.items()
}
print(f"Frases de la validación de PhraseBank: {len(validacion_07)}")
for nombre, proporcion in coinciden_con_el_07.items():
    print(f"  {nombre}: dice lo mismo que en el 07 en el {proporcion:.1%}")

Frases de la validación de PhraseBank: 967
  M1 diccionario: dice lo mismo que en el 07 en el 100.0%
  M2 Transformer: dice lo mismo que en el 07 en el 100.0%
  M3 BERT: dice lo mismo que en el 07 en el 100.0%


**Conclusión:** los tres modelos se cargan sin volver a entrenar y repiten **exactamente** lo que dijeron en el 07 (100 % de las frases de la validación). La copia de las capas y del vocabulario en `capas_keras.py` es fiel.

## 3. ¿Qué sentimiento tienen los titulares de FNSPID?

Los tres modelos puntúan los **192.069 titulares de FNSPID que mencionan a su empresa** (octubre 2016 – junio 2020, notebook 06). Los demás titulares de FNSPID son noticias en las que la empresa no aparece (listas de valores, noticias del mercado en general) y no cuentan para su sentimiento.

Algunos titulares mencionan a dos empresas y están dos veces, una con cada una. Cada texto distinto se puntúa una sola vez y el resultado se copia a sus filas.

**Salida:** una tabla con una fila por titular y modelo: empresa (`ticker`), fecha, titular, modelo, sentimiento (−1 negativo, 0 neutral, 1 positivo) y confianza. Se guarda en `data/cache/sentimiento_titulares.parquet`; si ya existe, se lee de disco. Puntuar todo lleva unos minutos.

In [9]:
RUTA_SENTIMIENTO = CARPETA_CACHE / "sentimiento_titulares.parquet"

noticias_fnspid = pd.read_parquet(CARPETA_CACHE / "noticias_fnspid.parquet")
titulares = noticias_fnspid.loc[noticias_fnspid["sobre_la_empresa"], ["ticker", "fecha", "titular"]]
textos = titulares["titular"].drop_duplicates().tolist()
print(f"Titulares que mencionan a su empresa: {len(titulares):,} · textos distintos: {len(textos):,}"
      f" · empresas: {titulares['ticker'].nunique()}")

if RUTA_SENTIMIENTO.exists():
    sentimiento = pd.read_parquet(RUTA_SENTIMIENTO)
    print("Sentimiento leído de disco")
else:
    partes = []
    for nombre, puntuar in MODELOS.items():
        puntos = puntuar(textos).assign(titular=textos, modelo=nombre)
        partes.append(titulares.merge(puntos, on="titular"))
        print(f"  {nombre}: hecho")
    sentimiento = pd.concat(partes, ignore_index=True)
    sentimiento["sentimiento"] = sentimiento["clase"].map(SENTIMIENTO_NUMERO)
    sentimiento = sentimiento[["ticker", "fecha", "titular", "modelo", "sentimiento", "confianza"]]
    sentimiento.to_parquet(RUTA_SENTIMIENTO, index=False)
    print("Sentimiento calculado y guardado")
sentimiento.head()

Titulares que mencionan a su empresa: 192,069 · textos distintos: 183,485 · empresas: 633
Sentimiento leído de disco


,ticker,fecha,titular,modelo,sentimiento,confianza
0,A,2016-11-10,"Leerink Swann Assumes Agilent Technologies, In...",M1 diccionario,1,0.1
1,A,2016-11-15,Agilent Technologies Sees FY17 Rev. $4.35B-$4....,M1 diccionario,0,0.0
2,A,2016-11-15,Agilent Technologies Reports Q4 EPS $0.59 vs. ...,M1 diccionario,0,0.0
3,A,2016-11-15,Agilent Technologies' (A) CEO Mike McMullen on...,M1 diccionario,0,0.0
4,A,2016-11-15,Agilent Technologies Inc. 2016 Q4 - Results - ...,M1 diccionario,0,0.0


**¿Cuántos titulares ve cada modelo como positivos, neutrales y negativos, y con qué confianza?** Como referencia, en PhraseBank el 59 % de las frases es neutral, el 28 % positiva y el 12,5 % negativa.

In [10]:
reparto = (pd.crosstab(sentimiento["modelo"], sentimiento["sentimiento"], normalize="index")
           .rename(columns={-1: "negativo", 0: "neutral", 1: "positivo"}) * 100)
reparto["confianza mediana"] = sentimiento.groupby("modelo")["confianza"].median()
reparto.round(1)

sentimiento,negativo,neutral,positivo,confianza mediana
modelo,,,,
M1 diccionario,13.8,74.6,11.6,0.0
M2 Transformer,7.9,67.2,24.9,0.8
M3 BERT,5.1,52.6,42.3,0.7


**¿Coinciden los modelos entre sí?** Proporción de titulares a los que cada par de modelos da el mismo sentimiento.

In [11]:
por_titular = sentimiento.pivot_table(index=["ticker", "fecha", "titular"], columns="modelo", values="sentimiento")
pares = [("M1 diccionario", "M2 Transformer"), ("M1 diccionario", "M3 BERT"), ("M2 Transformer", "M3 BERT")]
for a, b in pares:
    print(f"{a} y {b}: el mismo sentimiento en el {(por_titular[a] == por_titular[b]).mean():.1%} de los titulares")

M1 diccionario y M2 Transformer: el mismo sentimiento en el 60.0% de los titulares
M1 diccionario y M3 BERT: el mismo sentimiento en el 49.6% de los titulares
M2 Transformer y M3 BERT: el mismo sentimiento en el 58.8% de los titulares


Unos cuantos titulares al azar (con la semilla) con lo que dice cada modelo, para ver cómo leen los titulares:

In [12]:
with pd.option_context("display.max_colwidth", 110):
    display(por_titular.sample(12, random_state=SEMILLA).astype(int)
            .reset_index(["ticker", "fecha"], drop=True)
            .rename(columns={"M1 diccionario": "M1", "M2 Transformer": "M2", "M3 BERT": "M3"}))

modelo,M1,M2,M3
titular,,,
"Thinking about buying stock in Aurora Cannabis, American Express, PepsiCo, Wells Fargo, or Walmart?",0,0,0
Trimble +4.9% after $1.2B deal for Viewpoint,0,0,0
JPMorgan gets caught up in Venezuelan struggle,0,1,1
Are Investor Concerns On Monster Beverages Misplaced?,-1,0,0
Duke Energy launches 25M-share public offering,0,0,0
"Imperial Capital Maintains Outperform on Matador Resources, Raises Price Target to $25",1,1,1
FirstEnergy Invites Customers to Enter Holiday Lights Contest,0,0,0
"Evercore ISI Group Initiates Coverage On Incyte with In-Line Rating, Announces $135.00 Price Target",0,0,1
Why Is FactSet (FDS) Down 4.6% Since Last Earnings Report?,0,-1,0


**Conclusión: los tres modelos leen los mismos titulares de forma muy distinta.** Solo coinciden los tres en el 38 % de los titulares, y cada par coincide entre el 50 % y el 60 %.

- **M1 se queda casi siempre en neutral (75 %).** El 74 % de los titulares no tiene ninguna palabra del diccionario: es el mismo problema de cobertura que en PhraseBank, y peor, porque un titular tiene pocas palabras. Acierta cuando hay palabras de tono claras (*Outperform*, *Raises*), pero no ve *Down 4.6%*.
- **M2 tiene un reparto parecido al de PhraseBank,** con más positivos (25 %) que negativos (8 %). Es el único de los tres que lee *Why Is FactSet Down 4.6%* como negativo.
- **M3 ve muchos más positivos que los otros dos: el 42 % de los titulares.** Es su debilidad del 07 llevada a los titulares: allí ya fallaba sobre todo en las negativas. Aquí, de los titulares que M2 llama negativos, M3 llama positivos a la mitad (4,0 % de 7,9 %). Hay ejemplos de esto, como *JPMorgan gets caught up in Venezuelan struggle*, que tanto M2 como M3 dan por positivo.
- **Un límite común:** la estandarización del texto, la de los apuntes, borra los signos. *Trimble +4.9%* llega a las redes como *trimble 49*, y los tres lo dan por neutral aunque sea una subida.

Qué modelo lee mejor los titulares no se puede saber mirando solo estos repartos: lo dice el paso 4, con los 300 titulares etiquetados a mano.

## Comprobaciones

In [13]:
# Plantilla: 300 titulares distintos, ids del 1 al 300, como mucho 5 por empresa, y la misma muestra que se genera aquí
assert len(etiquetado) == TITULARES_A_MANO and etiquetado["id"].tolist() == list(range(1, TITULARES_A_MANO + 1))
assert etiquetado["titular"].is_unique
assert etiquetado["ticker"].value_counts().max() <= MAXIMO_POR_EMPRESA
assert etiquetado[["id", "ticker", "titular"]].equals(plantilla[["id", "ticker", "titular"]])
assert len(mal_escritos) == 0 and len(no_neutrales_ajenos) == 0
print("Plantilla: OK")

Plantilla: OK


In [14]:
# Modelos: los tres repiten exactamente lo que dijeron en el 07 sobre la validación de PhraseBank
assert keras.backend.backend() == "tensorflow"
assert all(proporcion == 1 for proporcion in coinciden_con_el_07.values()), coinciden_con_el_07
print("Modelos: OK")

Modelos: OK


In [15]:
# Sentimiento: una fila por titular y modelo, los mismos titulares que en FNSPID, valores válidos y sin huecos
assert len(sentimiento) == len(MODELOS) * len(titulares) == 3 * 192069
assert (sentimiento["modelo"].value_counts() == len(titulares)).all()
assert not sentimiento.duplicated(["ticker", "fecha", "titular", "modelo"]).any()
for nombre in MODELOS:
    del_modelo = sentimiento[sentimiento["modelo"] == nombre].sort_values(["ticker", "fecha", "titular"])
    assert del_modelo[["ticker", "fecha", "titular"]].reset_index(drop=True).equals(
        titulares.sort_values(["ticker", "fecha", "titular"]).reset_index(drop=True))
assert sentimiento["sentimiento"].isin([-1, 0, 1]).all() and sentimiento.notna().all().all()
assert sentimiento["confianza"].between(0, 1).all()
assert (sentimiento.loc[sentimiento["modelo"] != "M1 diccionario", "confianza"] >= 1 / 3 - 1e-6).all()
# M1 con confianza 0 (ninguna palabra del diccionario) es siempre neutral
m1_sin_palabras = sentimiento[(sentimiento["modelo"] == "M1 diccionario") & (sentimiento["confianza"] == 0)]
assert (m1_sin_palabras["sentimiento"] == 0).all()
print("Sentimiento de los titulares: OK")

Sentimiento de los titulares: OK
